# Classical Brain Tumour Classification Models

This notebook implements and evaluates:

- HOG and GLCM features with Support Vector Machine;
- HOG and GLCM features with Random Forest.

Five-fold stratified cross-validation will be performed using the fixed fold assignments created from the Training partition.

In [ ]:
%pip install scikit-image

In [21]:


from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import hog, graycomatrix, graycoprops

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "processed_data_cropped"
FOLDS_FILE_PATH = PROJECT_ROOT / "splits" / "five_fold_cross_validation.csv"

print("Project root:", PROJECT_ROOT)
print("Processed dataset exists:", DATA_DIR.exists())
print("Fold file exists:", FOLDS_FILE.exists())



Project root: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification
Processed dataset exists: True
Fold file exists: True


In [11]:
folds_df = pd.read_csv(FOLDS_FILE)

print("Fold file shape:", folds_df.shape)
print("Number of fold assignments", len(folds_df)) 
print("Fold file columns:", folds_df.columns)
print("Fold file head:\n", folds_df.head())


Fold file shape: (5600, 3)
Number of fold assignments 5600
Fold file columns: Index(['relative_path', 'class', 'fold'], dtype='str')
Fold file head:
                     relative_path   class  fold
0   Training/glioma/Tr-gl_100.png  glioma     1
1  Training/glioma/Tr-gl_1001.png  glioma     1
2  Training/glioma/Tr-gl_1003.png  glioma     1
3  Training/glioma/Tr-gl_1014.png  glioma     1
4  Training/glioma/Tr-gl_1015.png  glioma     1


## Cross-validation fold assignments

The fold-assignment file contains 5,600 rows and three columns. Each row represents one image from the original Training partition.

- `relative_path` records the image location relative to the processed dataset folder.
- `class` records the correct class label: glioma, meningioma, pituitary or no tumour.
- `fold` identifies which of the five folds will be used as the validation fold for that image.

For example, an image assigned to fold 1 will be used for validation during the first cross-validation iteration. Images assigned to folds 2–5 will be used for training during that iteration. The process is repeated so that every image is used for validation exactly once and for training in the other four iterations.

In [ ]:
# Verifying that class balance is maintained across folds

# Creating a frequency table to check the distribution of classes across folds
fold_distribution = pd.crosstab(folds_df['fold'], folds_df['class'])

fold_distribution["total"] = fold_distribution.sum(axis=1)
print("Fold distribution:\n", fold_distribution)

Fold distribution:
 class  glioma  meningioma  notumor  pituitary  total
fold                                                
1         280         280      280        280   1120
2         280         280      280        280   1120
3         280         280      280        280   1120
4         280         280      280        280   1120
5         280         280      280        280   1120


In [ ]:
# Check if all image paths exist

# Creating a new column in the DataFrame that contains the full image path by combining the DATA_DIR with the relative path from the CSV
folds_df["image_path"] = folds_df["relative_path"].apply(
    lambda relative_path: DATA_DIR / relative_path
)

# Checking if the image paths exist and collecting the missing ones
missing_paths = [
    image_path
    for image_path in folds_df["image_path"]
    if not image_path.exists()
]

print("Images checked:", len(folds_df))
print("Missing images:", len(missing_paths))
print("Fold file head:\n", folds_df.head())

In [ ]:
# Loading a sample image to verify that the paths are correct and the images can be read

# Selecting a sample image path from the DataFrame
sample_path = folds_df.loc[0, "image_path"]

# Loading the sample image using OpenCV in grayscale mode
sample_image = cv2.imread(
    str(sample_path),
    cv2.IMREAD_GRAYSCALE
)

print("Sample path:", sample_path)
print("Image loaded:", sample_image is not None)
print("Image shape:", sample_image.shape)
print("Data type:", sample_image.dtype)
print("Minimum pixel value:", sample_image.min())
print("Maximum pixel value:", sample_image.max())

plt.imshow(sample_image, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
# Computing HOG features for the sample image
sample_hog_features = hog(
    sample_image,
    orientations=9,
    pixels_per_cell=(16, 16),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    transform_sqrt=True,
    feature_vector=True
)

print("HOG feature shape:", sample_hog_features.shape)
print("Number of HOG features:", len(sample_hog_features))
print("Feature data type:", sample_hog_features.dtype)

HOG feature shape: (6084,)
Number of HOG features: 6084
Feature data type: float64


In [ ]:
# Testing GLCM feature extraction for the sample image

GLCM_LEVELS = 32

# Reducing the number of gray levels in the sample image to GLCM_LEVELS
sample_quantized_image = (sample_image / (256 / GLCM_LEVELS)).astype(np.uint8)

# Computing GLCM for the sample quantized image
sample_glcm = graycomatrix(
    sample_quantized_image,
    distances=[1, 2, 4],
    angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
    levels=GLCM_LEVELS,
    symmetric=True,
    normed=True
)

glcm_properties = [
    "contrast",
    "dissimilarity",
    "homogeneity",
    "energy",
    "correlation",
    "ASM"       
]

sample_glcm_features = []

# Computing GLCM properties for the sample image
for property_name in glcm_properties:
    property_values = graycoprops(sample_glcm, property_name)

    # Calculating the mean of the property values across angles for each distance
    mean_property_value = property_values.mean(axis=1)  # Mean across angles for each distance
    sample_glcm_features.extend(mean_property_value)
    
sample_glcm_features = np.array(sample_glcm_features)

# Printing the quantized pixel range
print("Quantized pixel range:",
    sample_quantized_image.min(),
    "to",
    sample_quantized_image.max())

# Printing the shape of the GLCM features
print("GLCM feature shape:",
    sample_glcm_features.shape)

Quantized pixel range: 0 to 30
GLCM feature shape: (18,)


In [27]:
# Combining HOG and GLCM features for the sample image

sample_combined_features = np.concatenate((sample_hog_features, sample_glcm_features))

print("HOG feature shape:", sample_hog_features.shape)
print("GLCM feature shape:", sample_glcm_features.shape)
print("Combined feature shape:", sample_combined_features.shape)

HOG feature shape: (6084,)
GLCM feature shape: (18,)
Combined feature shape: (6102,)


In [32]:
# Creating reusable functions for feature extraction

HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (16, 16)
HOG_CELLS_PER_BLOCK = (2, 2)

GLCM_LEVELS = 32
GLCM_DISTANCES = [1, 2, 4]
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]

GLCM_PROPERTIES = [
    "contrast",
    "dissimilarity",
    "homogeneity",
    "energy",
    "correlation",
    "ASM"
]

def extract_hog_features(image):
    """
    Extract HOG features from a grayscale image.

    Parameters:
    - image: Grayscale image (numpy array)

    Returns:
    - hog_features: 1D numpy array of HOG features
    """
    hog_features = hog(
        image,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True
    )
    
    return hog_features.astype(np.float32)

def extract_glcm_features(image):
    """
    Extract GLCM features from a grayscale image.

    Parameters:
    - image: Grayscale image (numpy array)

    Returns:
    - glcm_features: 1D numpy array of GLCM features
    """
    # Reducing the number of gray levels in the image to GLCM_LEVELS
    quantized_image = (image / (256 / GLCM_LEVELS)).astype(np.uint8)

    # Computing GLCM for the quantized image
    glcm = graycomatrix(
        quantized_image,
        distances=GLCM_DISTANCES,
        angles=GLCM_ANGLES,
        levels=GLCM_LEVELS,
        symmetric=True,
        normed=True
    )

    glcm_features = []

    # Computing GLCM properties for the image
    for property_name in GLCM_PROPERTIES:
        property_values = graycoprops(glcm, property_name)

        # Calculating the mean of the property values across angles for each distance
        mean_property_value = property_values.mean(axis=1)  # Mean across angles for each distance
        glcm_features.extend(mean_property_value)
    
    return np.array(glcm_features, dtype=np.float32)

def extract_combined_features(image):
    """
    Extract combined HOG and GLCM features from a grayscale image.

    Parameters:
    - image: Grayscale image (numpy array)

    Returns:
    - combined_features: 1D numpy array of combined HOG and GLCM features
    """
    hog_features = extract_hog_features(image)
    glcm_features = extract_glcm_features(image)
    
    combined_features = np.concatenate((hog_features, glcm_features))
    
    return combined_features

In [33]:
# Testing the reusable feature extraction functions on the sample image

test_hog_features = extract_hog_features(sample_image)
test_glcm_features = extract_glcm_features(sample_image)
test_combined_features = extract_combined_features(sample_image)

print("HOG shape:", test_hog_features.shape)
print("GLCM shape:", test_glcm_features.shape)
print("Combined shape:", test_combined_features.shape)

print("Combined data type:", test_combined_features.dtype)
print("All values finite:", np.isfinite(test_combined_features).all())

HOG shape: (6084,)
GLCM shape: (18,)
Combined shape: (6102,)
Combined data type: float32
All values finite: True


In [ ]:
# Extracting features for all images in the dataset

import time

number_of_images = len(folds_df)
number_of_features = len(extract_combined_features(sample_image))

all_features = np.empty((number_of_images, number_of_features), dtype=np.float32)

all_labels = folds_df["class"].to_numpy()
all_folds = folds_df["fold"].to_numpy()

start_time = time.perf_counter()

for position, (_, row) in enumerate(folds_df.iterrows()):
    image_path = row["image_path"]
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        raise ValueError(f"Image at {image_path} could not be loaded.")
    
    features = extract_combined_features(image)
    all_features[position] = extract_combined_features(image)
    
    if (position + 1) % 250 == 0:
        elapsed_time = time.perf_counter() - start_time
        print(f"Processed {position + 1}/{number_of_images} images in {elapsed_time:.2f} seconds.")


print("\nFeature extraction complete.")
print("Feature matrix shape:", all_features.shape)
print("Labels shape:", all_labels.shape)
print("Folds shape:", all_folds.shape)
print("Feature data type:", all_features.dtype)
print("All values finite:", np.isfinite(all_features).all())
print("Total time:", round(elapsed_time / 60, 2), "minutes")






In [37]:
# Saving the extracted features, labels, and folds to disk for later use

FEATURES_DIR = PROJECT_ROOT / "extracted_features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_FEATURES_FILE = (
    FEATURES_DIR
    / "hog_glcm_training_features.npz"
)

np.savez_compressed(
    TRAINING_FEATURES_FILE,
    features=all_features,
    labels=all_labels,
    folds=all_folds,
    relative_paths=folds_df["relative_path"].to_numpy()
)

print("Saved feature file:", TRAINING_FEATURES_FILE)
print("File exists:", TRAINING_FEATURES_FILE.exists())
print(
    "File size:",
    round(TRAINING_FEATURES_FILE.stat().st_size / (1024 ** 2), 2),
    "MB"
)

Saved feature file: c:\Users\kwsta\OneDrive\Desktop\Desetation project\brain-tumour-mri-classification\extracted_features\hog_glcm_training_features.npz
File exists: True
File size: 105.39 MB


In [ ]:
# Creating a scikit-learn cross-validation split using the folds information

from sklearn.model_selection import PredefinedSplit

# The CSV uses fold numbers 1-5
# PredefinedSplit expects fold numbers to be 0-indexed, so we subtract 1 from the fold numbers in the DataFrame

predefined_cv_split = PredefinedSplit(test_fold = all_folds.astype(int) - 1)

print("Number of cross-validation splits:", predefined_cv_split.get_n_splits())

for fold_number, (train_indices, validation_indices) in enumerate(predefined_cv_split.split(), start = 1):
    print(f"Fold {fold_number}:")
    print(f"  Training indices: {len(train_indices)} samples")
    print(f"  Validation indices: {len(validation_indices)} samples")

In [40]:
# Creating an SVM pipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_pipeline = Pipeline(
    steps=[
        ("feature_scaler", StandardScaler()),
        ("svm_classifier", SVC(kernel="rbf", C=1.0, gamma="scale"))
    ]
)
print(svm_pipeline)

Pipeline(steps=[('feature_scaler', StandardScaler()),
                ('svm_classifier', SVC())])


In [42]:
# Testing the SVM on fold 1

import time
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone

# Obtaining the first predefined cross-validation split (fold 1)
train_indices, validation_indices = next(predefined_cv_split.split())

X_train, X_val = all_features[train_indices], all_features[validation_indices]
y_train, y_val = all_labels[train_indices], all_labels[validation_indices]

# Creating an independent copy of the pipeline
fold_1_svm = clone(svm_pipeline)

start_time = time.perf_counter()

fold_1_svm.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

validation_predictions = fold_1_svm.predict(X_val)

validation_accuracy = accuracy_score(y_val, validation_predictions)

validation_macro_f1 =  f1_score(y_val, validation_predictions, average="macro")


print("Training samples:", X_train.shape)
print("Validation samples:", X_val.shape)
print("Training time:", round(training_time / 60, 2), "minutes")
print("Validation accuracy:", round(validation_accuracy, 4))
print("Validation macro F1-score:", round(validation_macro_f1, 4))


Training samples: (4480, 6102)
Validation samples: (1120, 6102)
Training time: 0.74 minutes
Validation accuracy: 0.8973
Validation macro F1-score: 0.8975


In [ ]:
# Evaluating the SVM using five-fold cross-validation

svm_fold_results = []

cross_validation_start_time = time.perf_counter()

for fold_number, (train_indices, validation_indices) in enumerate(
    predefined_cv_split.split(),
    start=1
):
    # Separating the training and validation data for the current fold
    X_train, X_val = (
        all_features[train_indices],
        all_features[validation_indices]
    )

    y_train, y_val = (
        all_labels[train_indices],
        all_labels[validation_indices]
    )

    # Creating an independent copy of the SVM pipeline
    fold_svm = clone(svm_pipeline)

    fold_start_time = time.perf_counter()

    # Training the SVM on the current training folds
    fold_svm.fit(X_train, y_train)

    fold_training_time = (
        time.perf_counter() - fold_start_time
    )

    # Making predictions on the current validation fold
    validation_predictions = fold_svm.predict(X_val)

    # Calculating the evaluation metrics
    validation_accuracy = accuracy_score(
        y_val,
        validation_predictions
    )

    validation_macro_f1 = f1_score(
        y_val,
        validation_predictions,
        average="macro"
    )

    # Saving the results for the current fold
    svm_fold_results.append(
        {
            "fold": fold_number,
            "accuracy": validation_accuracy,
            "macro_f1": validation_macro_f1,
            "training_time_minutes": fold_training_time / 60
        }
    )

    print(f"Fold {fold_number}:")
    print(
        "  Training time:",
        round(fold_training_time / 60, 2),
        "minutes"
    )
    print(
        "  Validation accuracy:",
        round(validation_accuracy, 4)
    )
    print(
        "  Validation macro F1-score:",
        round(validation_macro_f1, 4)
    )


total_cross_validation_time = (
    time.perf_counter() - cross_validation_start_time
)

# Converting the fold results into a DataFrame
svm_results_df = pd.DataFrame(svm_fold_results)

print("\nFive-fold cross-validation results:")
print(svm_results_df)

print(
    "\nMean validation accuracy:",
    round(svm_results_df["accuracy"].mean(), 4)
)

print(
    "Accuracy standard deviation:",
    round(svm_results_df["accuracy"].std(), 4)
)

print(
    "Mean validation macro F1-score:",
    round(svm_results_df["macro_f1"].mean(), 4)
)

print(
    "Macro F1-score standard deviation:",
    round(svm_results_df["macro_f1"].std(), 4)
)

print(
    "Total cross-validation time:",
    round(total_cross_validation_time / 60, 2),
    "minutes"
)